# 19. Validation of the `treatment_line_primary` rule

**Question:** is the priority rule `advanced > adjuvant > neoadjuvant > staging`, which decides the
`treatment_line_primary` component of the target's grain
(`ID_PLANO x treatment_line_primary x UF_PRESTADOR x MES`), fit for the purpose the project uses it for?

Notebook 13 applied the rule as an explicitly unvalidated placeholder, and the Article 6 review flagged
that the rule underpins the problem definition while carrying no validation of its own. This notebook
supplies that validation.

**It is read-only.** It recomputes the rule exactly as notebook 13 defines it, asserts it reproduces
notebook 13's counts, and changes nothing: no model is retrained, no artifact is rewritten, no rule is
altered. Every number in notebooks 09-18 is unaffected by running this notebook.

**Running the notebook:** Run All with Python 3.10+. The only inputs are
`data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet` (the Medicina de Grupo
base that notebooks 09-17 consume) and the reference crosswalk
`data/reference/tuss_sboc_crosswalk_breast_cancer.csv` (notebook 03's output).

**Five checks:**

| # | Check | What it decides |
| --- | --- | --- |
| 1 | Determinacy and coverage | Whether the rule is a total, deterministic function of the observed flag patterns |
| 2 | What the rule separates | Whether the resulting classes are distinguishable, or a cosmetic split |
| 3 | Reach into the target | Which parts of the target the rule can and cannot move |
| 4 | Partition effect | How much of the series structure the rule is responsible for |
| 5 | Clinical grounding | Whether the ordering is defensible against the SBOC crosswalk that produced the flags |


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__, "| python", sys.version.split()[0])

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks").is_dir() and (p / "documents").is_dir()), Path.cwd())
CLAIMS = ROOT / "data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet"
CROSSWALK = ROOT / "data/reference/tuss_sboc_crosswalk_breast_cancer.csv"
for path in (CLAIMS, CROSSWALK):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing input: {path.relative_to(ROOT)} -- run the preceding notebooks first.")

LINE_COLS = ["IND_LINE_STAGING", "IND_LINE_NEOADJUVANT", "IND_LINE_ADJUVANT", "IND_LINE_ADVANCED"]
LINE_NAMES = ["staging", "neoadjuvant", "adjuvant", "advanced"]  # ascending clinical acuity

df = pd.read_parquet(CLAIMS, columns=[
    "ID_PLANO", "UF_PRESTADOR", "MES", "QT_ITEM_EVENTO_INFORMADO", "IND_PACOTE_1",
    "category_drug", "category_procedure", "category_procedure_group", *LINE_COLS,
])
print("Claim item rows:", f"{len(df):,}")

pandas 3.0.5 | numpy 2.5.2 | python 3.14.3
Claim item rows: 70,004


## 1. The rule, restated and reproduced

The rule below is copied from notebook 13 without modification. The assertions are the guarantee that
this notebook validates the rule the project actually uses, rather than a re-derivation of it: the
flag-count distribution and the resulting class counts must equal the values notebook 13 printed.

In [2]:
# Verbatim from notebook 13: the rule under validation.
conditions = [
    df["IND_LINE_ADVANCED"] == 1,
    df["IND_LINE_ADJUVANT"] == 1,
    df["IND_LINE_NEOADJUVANT"] == 1,
    df["IND_LINE_STAGING"] == 1,
]
choices = ["advanced", "adjuvant", "neoadjuvant", "staging"]
df["treatment_line_primary"] = np.select(conditions, choices, default="unclassified")

n_lines_active = df[LINE_COLS].sum(axis=1)
flag_counts = n_lines_active.value_counts().sort_index()
class_counts = df["treatment_line_primary"].value_counts()

print("Events by number of simultaneously active IND_LINE_* flags:")
print(flag_counts)
print(f"\nRows with more than one line active: {(n_lines_active > 1).sum():,} "
      f"({(n_lines_active > 1).mean() * 100:.2f}%)")
print("\ntreatment_line_primary:")
print(class_counts)

# Reproduction guarantee against notebook 13's printed output.
assert flag_counts.to_dict() == {1: 232, 2: 24238, 3: 43979, 4: 1555}
assert class_counts.to_dict() == {"adjuvant": 42390, "neoadjuvant": 17564,
                                  "advanced": 10050}
print("\nOK -- reproduces notebook 13 exactly.")

Events by number of simultaneously active IND_LINE_* flags:
1      232
2    24238
3    43979
4     1555
Name: count, dtype: int64

Rows with more than one line active: 69,772 (99.67%)

treatment_line_primary:
treatment_line_primary
adjuvant       42390
neoadjuvant    17564
advanced       10050
Name: count, dtype: int64

OK -- reproduces notebook 12 exactly.


## 2. Check 1 - determinacy and coverage

A grain-defining rule must return exactly one class for every row, with no residual category and no
dependence on row order. The rule is a pure function of the four flags, so enumerating the observed flag
patterns characterises its behaviour completely.

In [3]:
pattern = df[LINE_COLS].astype(int).astype(str).agg("".join, axis=1)

patterns = pattern.value_counts().rename_axis("pattern").reset_index(name="rows")
patterns["flags_set"] = patterns["pattern"].map(
    lambda p: " + ".join(LINE_NAMES[i] for i in range(4) if p[i] == "1"))
patterns["assigned"] = patterns["pattern"].map(
    lambda p: next(LINE_NAMES[i] for i in (3, 2, 1, 0) if p[i] == "1"))
patterns["pct_rows"] = (patterns["rows"] / len(df) * 100).round(2)

print(f"Distinct flag patterns observed: {len(patterns)} of the 15 non-empty patterns possible")
display(patterns[["pattern", "flags_set", "assigned", "rows", "pct_rows"]])

assert (df["treatment_line_primary"] != "unclassified").all(), "residual class must stay empty"
assert (n_lines_active > 0).all(), "every row carries at least one line flag"
assert df.groupby(pattern, observed=True)["treatment_line_primary"].nunique().eq(1).all(), \
    "assignment must be a function of the flag pattern alone"
# The assigned class is always the highest-acuity flag that is actually set.
assert patterns.apply(
    lambda r: r["assigned"] == max(
        (LINE_NAMES[i] for i in range(4) if r["pattern"][i] == "1"), key=LINE_NAMES.index),
    axis=1).all()
print("OK -- total, deterministic, order-independent, and always the highest-acuity flag set.")

Distinct flag patterns observed: 9 of the 15 non-empty patterns possible


,pattern,flags_set,assigned,rows,pct_rows
0,1110,staging + neoadjuvant + adjuvant,adjuvant,38423,54.89
1,1100,staging + neoadjuvant,neoadjuvant,17564,25.09
2,0111,neoadjuvant + adjuvant + advanced,advanced,5556,7.94
3,0110,neoadjuvant + adjuvant,adjuvant,3889,5.56
4,0011,adjuvant + advanced,advanced,2769,3.96
5,1111,staging + neoadjuvant + adjuvant + advanced,advanced,1555,2.22
6,0001,advanced,advanced,154,0.22
7,0010,adjuvant,adjuvant,78,0.11
8,0101,neoadjuvant + advanced,advanced,16,0.02


OK -- total, deterministic, order-independent, and always the highest-acuity flag set.


**Result.** The rule is total (no `unclassified` rows), deterministic, and independent of row order:
the class depends only on the flag pattern, of which 9 of the 15 non-empty patterns occur. It always
returns the highest-acuity line the crosswalk actually flagged for that item, never a line the item was
not mapped to.

One composition finding falls out of the same table: **`staging` is never assigned**. It is flagged on
57,542 of the 70,004 rows, but never alone - a staging exam in this base always carries a neoadjuvant
flag as well - so the highest-acuity rule always resolves those rows to something later in the course.
The nominally four-level grain is a three-level one in practice, and the `staging` label should not be
described as a class this target contains.

## 3. Check 2 - what the rule actually separates

A partition rule earns its place only if the classes it produces are materially different. Section 4.3.9.7
of the documentation conjectured that the rule works statistically because it "happens to separate drugs
from imaging". That conjecture is testable directly.

In [4]:
df["item_category"] = np.select(
    [df["category_drug"] == 1, df["category_procedure"] == 1, df["category_procedure_group"] == 1],
    ["drug", "procedure", "procedure_group"], default="unmapped")

mix = (pd.crosstab(df["treatment_line_primary"], df["item_category"], normalize="index") * 100).round(1)
mix = mix.reindex([c for c in LINE_NAMES if c in mix.index])
print("Item category mix by assigned line (% of rows):")
display(mix)

measured = df["IND_PACOTE_1"].eq(0)
volume = df.loc[measured].groupby(
    "treatment_line_primary", observed=True)["QT_ITEM_EVENTO_INFORMADO"].sum()
profile = pd.DataFrame({
    "pct_rows": df["treatment_line_primary"].value_counts(normalize=True).mul(100).round(2),
    "pct_measured_volume": (volume / volume.sum() * 100).round(2),
}).reindex([c for c in LINE_NAMES if c in volume.index])
print("\nRow share vs measured-volume share:")
display(profile)

Item category mix by assigned line (% of rows):


item_category,drug,procedure,procedure_group
treatment_line_primary,,,
neoadjuvant,0.0,100.0,0.0
adjuvant,0.2,91.7,8.2
advanced,84.8,15.2,0.0



Row share vs measured-volume share:


,pct_rows,pct_measured_volume
treatment_line_primary,,
neoadjuvant,25.09,2.14
adjuvant,60.55,5.56
advanced,14.36,92.30


**Result.** The conjecture holds, and quantitatively.

- `advanced` is **84.8% drug** rows, `neoadjuvant` is **100% procedure**, and `adjuvant` is **91.7%
  procedure** with a further 8.2% procedure groups. The rule separates drug administration from imaging
  and procedural billing almost cleanly, which is why the resulting series are more homogeneous than an
  unsplit operator series.
- The split is also the difference between counting events and counting volume: `advanced` is **14.4% of
  rows but 92.3% of measured quantity**, because billed drug quantities are orders of magnitude larger
  than procedure quantities. A model trained on an unsplit series would be dominated by that one
  component with no way to see it.

This is the substantive justification for splitting at all, and it holds independently of whether the
label *names* are clinically exact.

## 4. Check 3 - how far the rule reaches into the target

The concern behind the review's finding is that an unvalidated rule sits underneath the problem
definition, so the decisive question is how far its influence reaches. The answer has two parts, and
only the first was obvious.

**Raw billed volume is invariant.** The rule relabels rows without creating, dropping or reweighting
any of them, so summing `SUM(QT_ITEM_EVENTO_INFORMADO)` back over the line dimension must return the
same operator x state x month quantity under any tie-break.

**The training target is not.** Notebook 13 does not train on raw billed volume. It applies a
measurement-quality rule *at group level*: a target group is usable only if none of its rows is a
package (`IND_PACOTE_1 = 1`), because a bundled row's quantity is unknown. Groups that are entirely
bundled stay null and groups that mix bundled with itemized rows give only a lower bound; both are
excluded from the training target and from the historical demand windows. Since `treatment_line_primary`
is part of the grouping key, the rule decides **which rows share a group**, and therefore which groups a
package row contaminates. Moving a package row into a clean group excludes that whole group; moving it
out can make a group eligible again.

Both parts are measured below: the roll-up identity, and the training target that survives the
measurement-quality rule under each rule.

In [5]:
def assign(priority_ascending):
    """Assign the highest-priority flag that is set; `priority_ascending` lists indices low to high."""
    out = np.full(len(df), "", dtype=object)
    for idx in priority_ascending:
        out = np.where(df[LINE_COLS[idx]].values == 1, LINE_NAMES[idx], out)
    return out

rules = {
    "current (adopted)": assign([0, 1, 2, 3]),   # advanced > adjuvant > neoadjuvant > staging
    "reverse": assign([3, 2, 1, 0]),             # staging > neoadjuvant > adjuvant > advanced
}
assert (rules["current (adopted)"] == df["treatment_line_primary"].values).all()

CELL = ["ID_PLANO", "UF_PRESTADOR", "MES"]
base = df.loc[measured].groupby(CELL, observed=True)["QT_ITEM_EVENTO_INFORMADO"].sum()

# Part A -- raw billed volume, summed back over the line dimension.
part_a = []
for name, labels in rules.items():
    per_line = df.assign(line=labels).loc[measured].groupby(
        CELL + ["line"], observed=True)["QT_ITEM_EVENTO_INFORMADO"].sum()
    rolled = per_line.groupby(level=[0, 1, 2]).sum()
    gap = float((rolled - base).abs().max())
    part_a.append({"line rule": name, "cells": len(rolled),
                   "max |rolled-up - unsplit|": gap, "raw volume": float(rolled.sum())})
    assert np.isclose(gap, 0.0), f"{name} must not move any cell total"

print("Part A -- raw billed volume by operator x state x month:")
display(pd.DataFrame(part_a))

# Part B -- notebook 13's measurement-quality rule, applied per target group.
GROUP = ["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]

def target_after_quality_rule(labels=None):
    """Notebook 13's rule: a group is trainable only if none of its rows is a package."""
    frame = df if labels is None else df.assign(treatment_line_primary=labels)
    keys = CELL if labels is None else GROUP
    grouped = frame.groupby(keys, observed=True).agg(
        volume=("QT_ITEM_EVENTO_INFORMADO", "sum"),
        rows=("IND_PACOTE_1", "size"),
        bundled_rows=("IND_PACOTE_1", "sum"),
    )
    trainable = grouped["bundled_rows"] == 0
    all_bundled = grouped["bundled_rows"] == grouped["rows"]
    return {
        "groups": len(grouped),
        "trainable groups": int(trainable.sum()),
        "all-bundled (null)": int(all_bundled.sum()),
        "partly bundled (lower bound)": int((~trainable & ~all_bundled).sum()),
        "training target volume": float(grouped.loc[trainable, "volume"].sum()),
    }

part_b = []
for name, labels in rules.items():
    part_b.append({"line rule": name, **target_after_quality_rule(labels)})
part_b.append({"line rule": "no line split", **target_after_quality_rule(None)})

adopted = part_b[0]["training target volume"]
for row in part_b:
    row["vs adopted"] = f"{(row['training target volume'] / adopted - 1) * 100:+.2f}%"
print("\nPart B -- training target after the measurement-quality rule:")
display(pd.DataFrame(part_b))

# The adopted rule must still reproduce notebook 13's published group counts.
assert part_b[0]["groups"] == 10302
assert part_b[0]["all-bundled (null)"] == 26
assert part_b[0]["partly bundled (lower bound)"] == 128
# Part A holds; Part B does not. Both are asserted so neither can silently change.
assert not np.isclose(part_b[1]["training target volume"], adopted), \
    "the reverse rule is expected to change the training target"
print("\nOK -- raw volume is invariant; the training target is not.")

Part A -- raw billed volume by operator x state x month:


,line rule,cells,max |rolled-up - unsplit|,raw volume
0,current (adopted),7315,0.0,822851.0
1,reverse,7315,0.0,822851.0



Part B -- training target after the measurement-quality rule:


,line rule,groups,trainable groups,all-bundled (null),partly bundled (lower bound),training target volume,vs adopted
0,current (adopted),10302,10148,26,128,821383.0,+0.00%
1,reverse,9299,9145,28,126,819848.0,-0.19%
2,no line split,7330,7177,15,138,737417.0,-10.22%



OK -- raw volume is invariant; the training target is not.


**Result: the rule is volume-preserving, but not target-preserving.**

*Part A.* Maximum deviation **0.0** across all **7,315** operator x state x month cells, under the
adopted rule and its exact reverse, on **822,851** raw billed items. Relabelling rows cannot move raw
billed volume, and this would hold for any relabelling whatsoever - which is exactly why it is not
sufficient evidence on its own.

*Part B.* Once notebook 13's measurement-quality rule is applied, the rule does move the target:

| Line rule | Groups | Trainable groups | Training target volume | vs adopted |
| --- | --- | --- | --- | --- |
| current (adopted) | 10,302 | 10,148 | 821,383 | — |
| reverse | 9,299 | 9,145 | 819,848 | **−0.19%** |
| no line split | 7,330 | 7,177 | 737,417 | **−10.22%** |

The adopted rule's group counts reproduce Section 4.3.6 exactly (10,302 groups, 26 entirely bundled,
128 partly bundled), so this is the same rule the persisted matrix was built with.

The mechanism is group contamination, not lost volume: a coarser partition puts more rows in each group,
so a single package row disqualifies more billed volume along with it. **The size of that effect depends
on the base.** On the Medicina de Grupo base the reverse rule costs only **0.19%** of the training
target, because packages are rare here (154 of 10,302 groups are affected at all); removing the line
split entirely still costs **10.22%**. On the earlier, unfiltered base the same comparison cost 13.68%,
so this is a property of how packages happen to fall across groups in a given base, not a constant of
the rule, and it must be re-measured whenever the population changes.

**What this bounds, and what it does not.** Demand *reported* at the operator x state x month grain is
untouched by the rule, so no operator-level demand figure in this project depends on it. What does
depend on it is which groups are measurable, and therefore the rows the supervised models train and are
scored on. Changing the rule later would require re-running notebooks 13 and 15-17, and the numbers
would move; Section 4.3.9.7 already measures what that costs the forecast.

## 5. Check 4 - how much partition structure the rule is responsible for

If the rule drove the series count, the model's behaviour would hinge on it. The comparison below shows
how far the panel geometry actually moves, and where the two rules agree.

In [6]:
geometry = []
for name, labels in rules.items():
    tmp = df.assign(line=labels)
    geometry.append({
        "line rule": name,
        "series (operator x line x UF)": tmp.groupby(
            ["ID_PLANO", "line", "UF_PRESTADOR"], observed=True).ngroups,
        "classes used": tmp["line"].nunique(),
    })
geometry.append({
    "line rule": "no line split",
    "series (operator x line x UF)": df.groupby(
        ["ID_PLANO", "UF_PRESTADOR"], observed=True).ngroups,
    "classes used": 1,
})
display(pd.DataFrame(geometry))

agreement = (rules["current (adopted)"] == rules["reverse"]).mean() * 100
unambiguous = (n_lines_active == 1).mean() * 100
print(f"Current vs reverse agreement: {agreement:.2f}% of rows")
print(f"Rows with a single line flag  : {unambiguous:.2f}% of rows")
assert np.isclose(agreement, unambiguous), "rules can only agree where the flags are unambiguous"
print("\nOK -- the two rules agree on exactly the unambiguous rows and nowhere else.")

,line rule,series (operator x line x UF),classes used
0,current (adopted),2305,3
1,reverse,1886,4
2,no line split,1443,1


Current vs reverse agreement: 0.33% of rows
Rows with a single line flag  : 0.33% of rows

OK -- the two rules agree on exactly the unambiguous rows and nowhere else.


**Result.** The adopted rule yields **2,305** series against **1,443** unsplit and **1,886** reversed:
the split adds roughly 60% more series than no split at all, so it is a real structural choice rather
than a formality. Current and reverse agree on **0.33%** of rows - exactly the 232 single-flag rows -
which confirms that the tie-break governs essentially every row, and therefore that the checks above
exercise the rule at full strength rather than on a rare edge case.

## 6. Check 5 - clinical grounding against the SBOC crosswalk

The flags come from notebook 03's TUSS/SBOC crosswalk, which maps a billed code to *every* treatment line
whose SBOC guideline cites it. Multi-mapping is the crosswalk's intended behaviour, not an error: TISS
Ambulatorial carries no diagnosis field, so a code appearing in more than one guideline genuinely cannot
be resolved from the claim alone. The question for the rule is which of the co-flagged lines to report.

In [7]:
crosswalk = pd.read_csv(CROSSWALK, encoding="utf-8-sig")
crosswalk["n_lines"] = crosswalk["treatment_line"].fillna("").map(
    lambda s: len([x for x in s.split(",") if x]))

print("Crosswalk entries:", len(crosswalk))
print("\nMapped lines per crosswalk entry, by item category:")
display(pd.crosstab(crosswalk["category"], crosswalk["n_lines"]))
print("\nMost frequent line combinations:")
display(crosswalk["treatment_line"].value_counts().head(8).rename("entries"))
print("\nSpecificity of crosswalk entries:")
display(crosswalk["specificity"].value_counts().rename("entries"))

Crosswalk entries: 397

Mapped lines per crosswalk entry, by item category:


n_lines,1,2,3,4
category,,,,
drug,129,55,130,8
procedure,9,43,18,3
procedure_group,0,2,0,0



Most frequent line combinations:


treatment_line
neoadjuvant,adjuvant,advanced            133
advanced                                 100
adjuvant,advanced                         49
adjuvant                                  34
neoadjuvant,adjuvant                      31
staging,neoadjuvant                       14
staging,neoadjuvant,adjuvant,advanced     11
staging,neoadjuvant,adjuvant               9
Name: entries, dtype: int64


Specificity of crosswalk entries:


specificity
specific           385
generic_staging     12
Name: entries, dtype: int64

**Result and the clinical reading.** 130 of the 322 drug entries are mapped to three lines at once
(`neoadjuvant, adjuvant, advanced`) - the systemic agents whose SBOC guidelines overlap by design. For
those, the adopted ordering reports `advanced`, that is, **the highest-acuity line consistent with the
evidence on the claim**.

Three properties make that ordering defensible for this deliverable:

1. **It is monotone in the SBOC disease course.** `staging -> neoadjuvant -> adjuvant -> advanced` is the
   guidelines' own progression, and the rule is simply "report the furthest point along it that the
   evidence admits". It is a stated convention with a direction, not an arbitrary tie-break.
2. **It never asserts unsupported evidence.** Check 1 showed the assigned line is always a flag the
   crosswalk actually set, so the label is never a claim the source data does not carry.
3. **It keeps high-cost volume together, and overstates the advanced share by construction.** Agents
   indicated for neoadjuvant, adjuvant *and* advanced disease are all reported as `advanced`, so the
   systemic-therapy volume stays in one class rather than being dispersed across earlier lines - useful
   for seeing where high-cost capacity is consumed, and what Check 2 observes in practice, since the
   `advanced` class is where the drug volume concentrates. The same construction inflates the advanced
   class: part of that volume belongs to patients in earlier lines. The figure must never be presented
   as the size of the advanced-disease segment. This matters commercially, because Enhertu sits in
   exactly this group of multi-line agents, so an unqualified `advanced` figure would overstate its
   addressable segment to the partner.

What this check does **not** establish is that an individual item labelled `advanced` was billed for a
patient in advanced disease. That is a per-patient clinical claim, and no evidence in this dataset can
support it - there is no diagnosis field and no beneficiary identifier.

## 7. Verdict

**The rule is validated for the use the project makes of it, with an explicit scope.**

| Check | Verdict |
| --- | --- |
| 1. Determinacy and coverage | **Pass.** Total, deterministic, order-independent; never asserts an unflagged line. `staging` is never assigned, so the grain is three-level in practice. |
| 2. What it separates | **Pass.** Separates drug from procedure billing almost cleanly (advanced 84.8% drug; neoadjuvant 100% procedure) and isolates the class carrying 92.3% of measured volume. |
| 3. Reach into the target | **Pass, with a boundary.** Raw billed volume is preserved exactly (0.0 across 7,315 cells), but through notebook 13's measurement-quality rule the rule does change the training target (−0.19% under the reverse rule on this base, −10.22% with no split). |
| 4. Partition effect | **Pass, quantified.** Governs 99.67% of rows and 2,305 of the panel's series; forecasting consequences measured in Section 4.3.9.7. |
| 5. Clinical grounding | **Pass as a stated convention.** Monotone in the SBOC disease course and evidence-bounded, while overstating the advanced share by construction. **Not** validated as a per-patient clinical label. |

**Scope of the validation.** `treatment_line_primary` is validated as a **series-partition convention**
for a utilization-volume proxy: checks 1-4 establish that it groups billed volume soundly, preserves
reported demand at the operator x state x month grain, and reaches the training target only through
notebook 13's measurement-quality rule, where its effect is measured rather than assumed away. It is
**not** validated as a clinical statement about individual patients, and the `advanced` class must not
be read as the size of the advanced-disease segment.

**What this closes.** The rule is no longer an undocumented placeholder underneath the problem
definition. Its behaviour is characterised, its reach is bounded and measured on both sides - invariant
for reported demand, quantified for the training target - and the cost of changing it later is known.

**What remains with the partner.** Confirming whether `advanced` is the reporting convention Daiichi
Sankyo wants for co-flagged systemic agents, knowing that it overstates the advanced share. Section
4.3.9.7 shows a clinically truer rule could make the forecast *worse* while making the labels *truer*.
Check 3 bounds what such a change would cost: reported operator-level demand is untouched, but the
training target moves, so the supervised notebooks would have to be re-run and their numbers would
change.